In [10]:
import pandas as pd

# Ruta al archivo (ajusta si es necesario)
file_path = "Especies_Conservadas_BBS-INIA_2026-02.xlsx"

# Cargar el archivo Excel
xls = pd.ExcelFile(file_path)

df_resumen = pd.read_excel(file_path, sheet_name="resumen")

#vamos a omitir las 3 primeras filas de df_resumen y la primera columna, que no contienen datos relevantes
df_resumen = df_resumen.iloc[3:, 1:]
#fila 4 tiene los nombres de las columnas, así que la asignamos como encabezado
df_resumen.columns = df_resumen.iloc[0]
#eliminamos la fila 4, que ahora es el encabezado
df_resumen = df_resumen.iloc[1:]


# Ver tabla
print(df_resumen)

3 Categoría de conservación  \
4                        EX   
5                        CR   
6                 EN + EN-R   
7                 VU + VU-R   
8          total amenazadas   
9        total clasificadas   

3 Especies de la flora chilena que han sido clasificadas  \
4                                                 17       
5                                                 98       
6                                                230       
7                                                171       
8                                                516       
9                                                675       

3 total clasificadas especies con semillas especies almacenadas en el BBS   \
4                                       15                               1   
5                                       81                              17   
6                                      202                              77   
7                                      143       

In [14]:
#vamos a hacer una tabla mas ordenadas con df_resumen
    

# df = tu dataframe actual
tabla = df_resumen.copy()


# 1) Aplanar columnas por si vienen como MultiIndex y normalizar espacios
tabla.columns = [
    " ".join([str(x) for x in c if str(x) != "nan"]).strip() if isinstance(c, tuple) else str(c).strip()
    for c in tabla.columns
]
tabla.columns = tabla.columns.str.replace(r"\s+", " ", regex=True)

# 2) Ver columnas reales (para depurar)
print(tabla.columns.tolist())

# 3) Renombrar con coincidencias más flexibles
rename_map = {}
for col in tabla.columns:
    c = col.lower()
    if "categor" in c and "conserv" in c:
        rename_map[col] = "Categoría"
    elif "flora chilena" in c and "clasific" in c:
        rename_map[col] = "Flora clasificada"
    elif "almacenadas en el bbs" in c or ("semillas" in c and "bbs" in c):
        rename_map[col] = "Con semillas en BBS"
    elif "amenazadas con semillas" in c:
        rename_map[col] = "% amenazadas con semillas"

tabla = tabla.rename(columns=rename_map)

# 4) Convertir solo las que realmente existan
num_cols = [c for c in ["Flora clasificada", "Con semillas en BBS", "% amenazadas con semillas"] if c in tabla.columns]

for c in num_cols:
    tabla[c] = (
        tabla[c].astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )
    tabla[c] = pd.to_numeric(tabla[c], errors="coerce")

if "% amenazadas con semillas" in tabla.columns:
    tabla["% amenazadas con semillas"] = tabla["% amenazadas con semillas"].round(1)

# 5) Mostrar
fmt = {}
if "Flora clasificada" in tabla.columns: fmt["Flora clasificada"] = "{:,.0f}"
if "Con semillas en BBS" in tabla.columns: fmt["Con semillas en BBS"] = "{:,.0f}"
if "% amenazadas con semillas" in tabla.columns: fmt["% amenazadas con semillas"] = "{:.1f}%"

display(
    tabla.style
    .hide(axis="index")
    .format(fmt)  # usa tu diccionario fmt actual
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#e6e6e6"),
                ("color", "#222"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("border", "1px solid #d0d0d0")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("padding", "6px"),
                ("border", "1px solid #e0e0e0")
            ]
        }
    ])
    .set_caption("Estado de conservación y representación en BBS")
)



['Categoría de conservación', 'Especies de la flora chilena que han sido clasificadas', 'total clasificadas especies con semillas', 'especies almacenadas en el BBS', '% de las amenazadas con semillas (no considera helechos ni briófitas)']


Categoría,Flora clasificada,total clasificadas especies con semillas,Con semillas en BBS,% amenazadas con semillas
EX,17,15,1,6.7%
CR,98,81,17,21.0%
EN + EN-R,230,202,77,38.1%
VU + VU-R,171,143,84,58.7%
total amenazadas,516,441,179,40.6%
total clasificadas,675,541,244,45.1%
